In [ ]:
# Python libraries
import time
import os
import numpy as np

# Pynq libraries
from pynq import MMIO
from pynq import Overlay
from pynq import allocate

In [ ]:
# Loading the bitstream
BITSTREAM_FILE = "/home/xilinx/pl/design_1_wrapper.bit"
mod_time = os.path.getmtime(BITSTREAM_FILE)
print("Last modified:", time.ctime(mod_time))
ol = Overlay(BITSTREAM_FILE)


In [ ]:
# Finding the base address of the qpi flash ip
print("All IP keys are: ", ol.ip_dict.keys())
FLASH_IP = [name for name in ol.ip_dict.keys() if "flash_axis" in name]
DMA_IP   = [name for name in ol.ip_dict.keys() if "axi_dma" in name]

if not FLASH_IP:
    print("Could not find flash IP!\nThese are the IPs:")
    print(ol.ip_dict.keys())
else:
    FLASH_IP_NAME = FLASH_IP[0]  # assuming one match
    BASEADDR = ol.ip_dict[FLASH_IP_NAME]['phys_addr']
    print(f"Base address of {FLASH_IP_NAME}: {hex(BASEADDR)}")
    
if not DMA_IP:
    print("Could not find AXI_DMA IP!\nThese are the IPs:")
    print(ol.ip_dict.keys())
else:
    DMA_IP_NAME = DMA_IP[0]  # assuming one match
    BASEADDR = ol.ip_dict[DMA_IP_NAME]['phys_addr']
    print(f"Base address of {DMA_IP_NAME}: {hex(BASEADDR)}")
    

CMD_REG_OFF   = 0x0
STAT_REG_OFF  = 0x8
STAT_REG_MAX  = 0x0
CMD_WEN_MASK  = 0x1
STAT_BIT_MASK = 0x1
STAT_WIP_MASK = 0x4

mmio = MMIO(BASEADDR, 0x1000)

In [ ]:
# Function cells
def wait_ready(mask=STAT_BIT_MASK, timeout=2.0):
    start = time.time()
    while True:
        ready = mmio.read(STAT_REG_OFF) & mask
        if ready:
            print("FLASH read!")
            return True
        if time.time() - start > timeout:
            print("Timeout waiting for flash ready")
            return False
        time.sleep(0.001)

def flash_read(addr, length):
    # Disable write
    mmio.write(CMD_REG_OFF, 0x0)

    # Prepare TX buffer (address + length)
    tx_buf = allocate(shape=(2,), dtype=np.uint32)
    tx_buf[0] = addr
    tx_buf[1] = length

    # Prepare RX buffer for incoming data
    rx_buf = allocate(shape=(length,), dtype=np.uint32)

    # Start DMA transfers
    dma.sendchannel.transfer(tx_buf)
    dma.recvchannel.transfer(rx_buf)

    dma.sendchannel.wait()
    dma.recvchannel.wait()

    print(f"Read {length} words from {hex(addr)}")
    return rx_buf


def flash_write(addr, data_words):
    # Enable write
    mmio.write(CMD_REG_OFF, CMD_WEN_MASK)

    total_len = 1 + len(data_words)

    # Prepare TX buffer (address + data)
    tx_buf = allocate(shape=(total_len,), dtype=np.uint32)
    tx_buf[0] = addr
    tx_buf[1:] = data_words

    # Send via DMA
    dma.sendchannel.transfer(tx_buf)
    dma.sendchannel.wait()

    # Wait for flash write completion
    wait_ready(mask=STAT_WIP_MASK, timeout=5.0)

    # Disable write
    mmio.write(CMD_REG_OFF, 0x0)

    print(f"Wrote {len(data_words)} words to {hex(addr)}")

In [ ]:
# Small
wait_ready()
flash_read(0x00004000, 20)
flash_read(0x01FF0030, 1)

flash_write(0x00001111, [0xd0bad0ba])
flash_read(0x00001111, 4)

flash_write(0x01BABABA, [0xb000b555, 0x65156aaa, 0x01234567, 0x890abcde, 0xfedcba09])
flash_read(0x01BABABA, 32)
